# 00 先理解大模型评估：不是先跑分

这一节先回答一个最基础的问题：**大模型评估到底在评什么？**

不要一开始就想“我要跑哪个榜”。榜单、测试集、loss、人工复核都只是评估信号。我们真正想知道的是：模型在目标任务里能不能稳定做出我们需要的行为。

## 1. 评估不是一个分数

一个大模型的能力通常要分层看：

```text
基础语言能力：能不能读懂问题、中文是否流畅
任务能力：能不能完成你定义的任务
安全边界：什么时候该回答，什么时候该拒绝危险细节
稳定性：换个问法是否还表现正常
回归保护：新能力提升后，旧能力有没有变差
工程可复现：同一批 prompt、同一配置，结果能不能复盘
```

所以面试时不要说“我跑了一个分数”。更好的说法是：

> 我把目标能力拆成若干行为维度，建立 held-out 评测集，先评估 baseline，定位 badcase，再用 badcase 反推数据构造和微调策略。

## 2. 常见评估信号

| 信号 | 看什么 | 优点 | 局限 |
|---|---|---|---|
| train/eval loss | 模型拟合训练/验证文本的程度 | 自动、便宜、训练时就有 | 不等于真实回答质量 |
| 公开 benchmark | MMLU、C-Eval、CMMLU 等通用能力 | 横向可比 | 不一定覆盖你的产品任务 |
| 固定 prompt 行为评估 | 对同一批问题生成回答并比较 | 能发现具体 badcase | 需要设计好题目和评分 |
| 规则评分 | 命中必要元素、避开禁忌元素 | 可解释、可复现 | 对语义理解有限 |
| 人工复核 | 人读回答，判断质量 | 最贴近真实体验 | 慢、主观，需要 rubric |
| LLM-as-judge | 用另一个模型当评委 | 快、覆盖语义 | judge 也会偏，需要校准 |

这个项目先用固定 prompt + 规则评分 + 人工抽查，因为这最适合你现在面试前快速掌握。

## 3. “用自己评估自己”对吗？

不对。如果我们让同一个模型生成答案，又让同一个模型给自己打分，这叫自评，偏差很大。

但我们之前说的“生成四路回答”不是让模型自己评估自己，而是：

```text
base 模型生成回答
public-SFT 生成回答
custom-SFT 生成回答
DPO adapter 生成回答
然后由同一套外部规则 / 人工 rubric / judge 去评分
```

模型只是被测对象，不是裁判。裁判是我们写死的评测集、评分规则和人工复核。

## 4. Stage 7 的评估问题

Stage 7 的目标不是“让模型更安全”这么泛，而是一个可评估的问题：

```text
安全敏感场景下的大模型帮助能力提升：
从过度拒答到有边界地提供有效帮助
```

我们要看两类失败：

```text
过度拒答：用户需要合法安全帮助，但模型只拒绝，不给可执行支持。
危险放开：用户请求危险细节，模型给了不该给的操作性建议。
```

这就是后面设计评测集和数据集的核心。